# **Data Preprocessing**
Ta sẽ thực hiện tiền xử lý dữ liệu để tối ưu quá trình huấn luyện mô hình phân loại Softmax Regression

## *Import các thư viện cần thiết*

In [49]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
from IPython.display import display, Markdown, HTML
import textwrap

## *Load dữ liệu*

In [50]:
try:
    raw_data = np.genfromtxt('../data/raw/BankChurners.csv', delimiter=',', names=True, dtype=None, encoding='utf-8')
    display(Markdown(f"### Đã tải dữ liệu thành công!"))
    display(Markdown(f"- Tổng số mẫu ban đầu: **{len(raw_data)}**"))
    display(Markdown(f"- Các cột (features): {raw_data.dtype.names}"))
except Exception as e:
    display(Markdown(f"### Lỗi khi tải file: {e}"))
    raw_data = None

### Đã tải dữ liệu thành công!

- Tổng số mẫu ban đầu: **10127**

- Các cột (features): ('CLIENTNUM', 'Attrition_Flag', 'Customer_Age', 'Gender', 'Dependent_count', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal', 'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio', 'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1', 'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2')

## *1. Kiểm tra xem có các quan sát nào bị trùng ID trong tập dữ liệu hay không*
* Nếu có, ta sẽ thực hiện loại bỏ trùng

In [51]:
if raw_data is not None:
    client_nums = raw_data['CLIENTNUM']

    # Tìm các ID trùng và số lần xuất hiện
    unique_ids, counts = np.unique(client_nums, return_counts=True)
    duplicate_ids = unique_ids[counts > 1]

    display(Markdown("### Kiểm tra trùng lặp `CLIENTNUM`"))

    if len(duplicate_ids) > 0:
        # Thông báo số lượng
        display(Markdown(f" **Phát hiện `{len(duplicate_ids)}` mã khách hàng (CLIENTNUM) bị trùng.**"))
        display(Markdown(f"*Hiển thị chi tiết mẫu cho 5 ID trùng đầu tiên:*"))
        
        # Lấy dữ liệu mẫu để hiển thị
        ids_to_show = duplicate_ids[:5] 
        duplicate_rows = []
        
        # Thu thập các dòng dữ liệu bị trùng
        for uid in ids_to_show:
            indices = np.where(client_nums == uid)[0]
            for idx in indices:
                duplicate_rows.append(raw_data[idx])

        if len(duplicate_rows) > 0:
            # Các cột muốn hiển thị
            show_cols = ['CLIENTNUM', 'Attrition_Flag', 'Customer_Age', 'Gender', 'Credit_Limit']
            
            header = "| " + " | ".join(show_cols) + " |"
            separator = "| " + " | ".join(["---"] * len(show_cols)) + " |"
            
            rows_str = ""
            for row in duplicate_rows:
                vals = [str(row[col]) for col in show_cols]
                rows_str += "| " + " | ".join(vals) + " |\n"
            
            full_table = f"{header}\n{separator}\n{rows_str}"
            display(Markdown(full_table))
            
    else:
        display(Markdown("**Không phát hiện `CLIENTNUM` nào bị trùng.**"))

### Kiểm tra trùng lặp `CLIENTNUM`

**Không phát hiện `CLIENTNUM` nào bị trùng.**

## *2. Chuyển đổi kiểu dữ liệu phù hợp đối với các đặc trưng*

In [52]:
def summary_dataset(data):
    num_samples = data.shape[0]
    feature_names = data.dtype.names
    
    if not feature_names:
        print("Lỗi: Dataset không có tên trường (structured array).")
        return

    num_features = len(feature_names)
    
    rows_list = []
    for i, name in enumerate(feature_names):
        dtype_val = data.dtype[name]
        rows_list.append(f"| {i} | **{name}** | `{dtype_val}` |")
    
    rows_content = "\n".join(rows_list)

    md_lines = [
        "### DATASET SUMMARY",
        "---",
        f"- **Số lượng mẫu:** `{num_samples:,}`",
        f"- **Số lượng features:** `{num_features}`",
        "", 
        "<br>",
        "",
        "### CHI TIẾT CÁC ĐẶC TRƯNG (FEATURES)",
        "| ID | Tên đặc trưng | Kiểu dữ liệu (Dtype) |",  
        "|:---|:---|:---|",                              
        rows_content                                     
    ]
    
    final_md = "\n".join(md_lines)

    display(Markdown(final_md))


### 2.1. Trước khi chuyển đổi

In [53]:
summary_dataset(raw_data)

### DATASET SUMMARY
---
- **Số lượng mẫu:** `10,127`
- **Số lượng features:** `23`

<br>

### CHI TIẾT CÁC ĐẶC TRƯNG (FEATURES)
| ID | Tên đặc trưng | Kiểu dữ liệu (Dtype) |
|:---|:---|:---|
| 0 | **CLIENTNUM** | `int64` |
| 1 | **Attrition_Flag** | `<U19` |
| 2 | **Customer_Age** | `int64` |
| 3 | **Gender** | `<U3` |
| 4 | **Dependent_count** | `int64` |
| 5 | **Education_Level** | `<U15` |
| 6 | **Marital_Status** | `<U10` |
| 7 | **Income_Category** | `<U16` |
| 8 | **Card_Category** | `<U10` |
| 9 | **Months_on_book** | `int64` |
| 10 | **Total_Relationship_Count** | `int64` |
| 11 | **Months_Inactive_12_mon** | `int64` |
| 12 | **Contacts_Count_12_mon** | `int64` |
| 13 | **Credit_Limit** | `float64` |
| 14 | **Total_Revolving_Bal** | `int64` |
| 15 | **Avg_Open_To_Buy** | `float64` |
| 16 | **Total_Amt_Chng_Q4_Q1** | `float64` |
| 17 | **Total_Trans_Amt** | `int64` |
| 18 | **Total_Trans_Ct** | `int64` |
| 19 | **Total_Ct_Chng_Q4_Q1** | `float64` |
| 20 | **Avg_Utilization_Ratio** | `float64` |
| 21 | **Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1** | `float64` |
| 22 | **Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2** | `float64` |

### 2.2. Sau khi chuyển đổi và làm sạch các kiểu dữ liệu là string

In [54]:
display(Markdown("### 2. Chuyển đổi kiểu dữ liệu & Làm sạch String"))

# Xác định dtype mới
new_dtype_list = []
original_dtypes = raw_data.dtype

# Duyệt để tạo cấu trúc dtype mới
for name in original_dtypes.names:
		current_type = original_dtypes[name]
		if np.issubdtype(current_type, np.number):
				new_dtype_list.append((name, current_type))
		else:
				# String chuyển thành U50
				new_dtype_list.append((name, 'U50'))

# Tạo mảng rỗng với cấu trúc mới
final_data = np.empty(raw_data.shape, dtype=new_dtype_list)

# Copy và Làm sạch dữ liệu
for name in original_dtypes.names:
		# Lấy dữ liệu cột hiện tại
		col_data = raw_data[name]
		
		if np.issubdtype(original_dtypes[name], np.number):
				final_data[name] = col_data
		else:
				clean_col = np.char.replace(col_data.astype(str), '"', '')
				final_data[name] = clean_col
				
display(Markdown("**Đã chuyển đổi kiểu dữ liệu và loại bỏ dấu ngoặc kép thừa.**"))
summary_dataset(final_data)

### 2. Chuyển đổi kiểu dữ liệu & Làm sạch String

**Đã chuyển đổi kiểu dữ liệu và loại bỏ dấu ngoặc kép thừa.**

### DATASET SUMMARY
---
- **Số lượng mẫu:** `10,127`
- **Số lượng features:** `23`

<br>

### CHI TIẾT CÁC ĐẶC TRƯNG (FEATURES)
| ID | Tên đặc trưng | Kiểu dữ liệu (Dtype) |
|:---|:---|:---|
| 0 | **CLIENTNUM** | `int64` |
| 1 | **Attrition_Flag** | `<U50` |
| 2 | **Customer_Age** | `int64` |
| 3 | **Gender** | `<U50` |
| 4 | **Dependent_count** | `int64` |
| 5 | **Education_Level** | `<U50` |
| 6 | **Marital_Status** | `<U50` |
| 7 | **Income_Category** | `<U50` |
| 8 | **Card_Category** | `<U50` |
| 9 | **Months_on_book** | `int64` |
| 10 | **Total_Relationship_Count** | `int64` |
| 11 | **Months_Inactive_12_mon** | `int64` |
| 12 | **Contacts_Count_12_mon** | `int64` |
| 13 | **Credit_Limit** | `float64` |
| 14 | **Total_Revolving_Bal** | `int64` |
| 15 | **Avg_Open_To_Buy** | `float64` |
| 16 | **Total_Amt_Chng_Q4_Q1** | `float64` |
| 17 | **Total_Trans_Amt** | `int64` |
| 18 | **Total_Trans_Ct** | `int64` |
| 19 | **Total_Ct_Chng_Q4_Q1** | `float64` |
| 20 | **Avg_Utilization_Ratio** | `float64` |
| 21 | **Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1** | `float64` |
| 22 | **Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2** | `float64` |

## *3. Xoá các cột dư thừa*
Bao gồm: `CLIENTNUM`, `Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1`, `Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2`

In [55]:
all_features = list(final_data.dtype.names)
keep_features = all_features[1:-2]
final_data = final_data[keep_features]
display(Markdown(f"**Danh sách cột mới:** {final_data.dtype.names}"))

**Danh sách cột mới:** ('Attrition_Flag', 'Customer_Age', 'Gender', 'Dependent_count', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal', 'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio')

## *4. Biến đổi dữ liệu để phục vụ cho việc huấn luyện mô hình*

### 4.1. Số hoá nhãn mục tiêu `Attrition_Flag`
`Attrited Customer` --> 1, `Existing Customer` --> 0

In [56]:
names = keep_features
y = np.where(final_data['Attrition_Flag'] == 'Attrited Customer', 1, 0).astype(int)

Trước khi xử lý tiếp đặc trưng numeric và categorical, ta sẽ phân loại chúng

In [57]:
# Danh sách các cột phân loại (Categorical)
cat_cols = ['Gender', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category']
# Danh sách các cột số (Numerical)
num_cols = [n for n in names if n not in cat_cols and n != 'Attrition_Flag']

cat_cols, num_cols

(['Gender',
  'Education_Level',
  'Marital_Status',
  'Income_Category',
  'Card_Category'],
 ['Customer_Age',
  'Dependent_count',
  'Months_on_book',
  'Total_Relationship_Count',
  'Months_Inactive_12_mon',
  'Contacts_Count_12_mon',
  'Credit_Limit',
  'Total_Revolving_Bal',
  'Avg_Open_To_Buy',
  'Total_Amt_Chng_Q4_Q1',
  'Total_Trans_Amt',
  'Total_Trans_Ct',
  'Total_Ct_Chng_Q4_Q1',
  'Avg_Utilization_Ratio'])

### 4.2. Dữ liệu numeric

Để mô hình Softmax Regression hội tụ nhanh chóng, ta sẽ đưa miền giá trị của các đặc trưng numeric về cùng miền giá trị với phân phối chuẩn bằng cách chuẩn hoá theo Z-score

$$
X = \frac {X-mean}{std}
$$

In [58]:
X_nums = []
for col in num_cols:
		col_data = final_data[col].astype(float)
		mean = np.mean(col_data)
		std = np.std(col_data)
		# Tránh chia cho 0 nếu std = 0
		if std == 0: std = 1 
		
		normalized_col = (col_data - mean) / std
		X_nums.append(normalized_col)

X_nums = np.column_stack(X_nums)

X_nums[:5]

array([[-1.65405580e-01,  5.03368127e-01,  3.84620878e-01,
         7.63942609e-01, -1.32713603e+00,  4.92403766e-01,
         4.46621903e-01, -4.73422218e-01,  4.88970818e-01,
         2.62349444e+00, -9.59706574e-01, -9.73895182e-01,
         3.83400260e+00, -7.75882235e-01],
       [ 3.33570383e-01,  2.04319867e+00,  1.01071482e+00,
         1.40730617e+00, -1.32713603e+00, -4.11615984e-01,
        -4.13666521e-02, -3.66666822e-01, -8.48598788e-03,
         3.56329284e+00, -9.16432607e-01, -1.35734038e+00,
         1.26085729e+01, -6.16275655e-01],
       [ 5.83058365e-01,  5.03368127e-01,  8.96451285e-03,
         1.20579050e-01, -1.32713603e+00, -2.21965548e+00,
        -5.73697797e-01, -1.42685834e+00, -4.45658333e-01,
         8.36721381e+00, -7.40981694e-01, -1.91120566e+00,
         6.80786367e+00, -9.97154993e-01],
       [-7.89125535e-01,  1.27328340e+00, -2.41473064e-01,
        -5.22784510e-01,  1.64147829e+00, -1.31563573e+00,
        -5.85251078e-01,  1.66168570e+00, -7.

### 4.3. Dữ liệu categorical

**Chiến lược:** Ta sẽ thực hiện One-Hot encoding cho từng đặc trưng categorical thành các cột riêng biệt nơi mỗi cột là một giá trị của cột đặc trưng đó

Vì các giá trị unique của các đặc trưng categorical là không nhiều (<= 7) nên ta có thể thực hiện tốt việc encoding trên

In [59]:
X_cats = []
for col in cat_cols:
		col_data = final_data[col]
		unique_vals = np.unique(col_data)
		
		# Với mỗi giá trị unique, tạo 1 cột nhị phân (0/1)
		for val in unique_vals:
				# Tạo cột: 1 nếu bằng val, 0 nếu không
				binary_col = np.where(col_data == val, 1, 0).astype(int)
				X_cats.append(binary_col)
				
X_cats = np.column_stack(X_cats)

### 4.4. Lưu lại dữ liệu đã tiền xử lý

In [60]:
X_final = np.hstack((X_nums, X_cats))

header_names = list(num_cols)

for col in cat_cols:
		col_data = final_data[col]
		unique_vals = np.unique(col_data)
		
		for val in unique_vals:
				clean_val = str(val).strip() 
				new_col_name = f"{col}_{clean_val}"
				header_names.append(new_col_name)
				
header_names.append("Attrition_Flag")
y_reshaped = y.reshape(-1, 1)
full_data_matrix = np.hstack((X_final, y_reshaped))
header_str = ",".join(header_names)

file_path = '../data/processed/bank_churners_preprocessed.csv'
column_formats = ['%f'] * X_final.shape[1] + ['%d']
try:
		np.savetxt(file_path, 
								full_data_matrix, 
								delimiter=",", 
								header=header_str, 
								comments="", 
								fmt=column_formats)
		
		display(Markdown(f"### Đã lưu file thành công: `{file_path}`"))
		display(Markdown(f"- **Số lượng cột:** {len(header_names)}"))
		display(Markdown(f"- **Danh sách cột:** {header_names}"))
  
except Exception as e:
        display(Markdown(f"### Lỗi khi lưu file: {e}"))

### Đã lưu file thành công: `../data/processed/bank_churners_preprocessed.csv`

- **Số lượng cột:** 38

- **Danh sách cột:** ['Customer_Age', 'Dependent_count', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal', 'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio', 'Gender_F', 'Gender_M', 'Education_Level_College', 'Education_Level_Doctorate', 'Education_Level_Graduate', 'Education_Level_High School', 'Education_Level_Post-Graduate', 'Education_Level_Uneducated', 'Education_Level_Unknown', 'Marital_Status_Divorced', 'Marital_Status_Married', 'Marital_Status_Single', 'Marital_Status_Unknown', 'Income_Category_$120K +', 'Income_Category_$40K - $60K', 'Income_Category_$60K - $80K', 'Income_Category_$80K - $120K', 'Income_Category_Less than $40K', 'Income_Category_Unknown', 'Card_Category_Blue', 'Card_Category_Gold', 'Card_Category_Platinum', 'Card_Category_Silver', 'Attrition_Flag']